Note: ICP dataset is downloaded from the ICP website. Add into files before running code.

In [1]:
import pandas as pd
from pathlib import Path

icp = pd.read_csv(Path("icp_raw.csv"))
print(icp.shape)
print(icp.columns.tolist())
print(icp.head(8).to_string())

(1279, 8)
['Country Name', 'Country Code', 'Classification Name', 'Classification Code', 'Series Name', 'Series Code', '2017 [YR2017]', '2021 [YR2021]']
  Country Name Country Code                           Classification Name Classification Code                                                  Series Name Series Code  2017 [YR2017]  2021 [YR2021]
0       Canada          CAN               Price level index (World = 100)               PX.WL                                            1107000:TRANSPORT     1107000     125.787848     128.399849
1       Canada          CAN               Price level index (World = 100)               PX.WL                                   1107300:TRANSPORT SERVICES     1107300     261.096078     267.786445
2       Canada          CAN               Price level index (World = 100)               PX.WL                               1111000:RESTAURANTS AND HOTELS     1111000     152.484684     150.834531
3       Canada          CAN               Price level index

In [2]:
icp = pd.read_csv("icp_raw.csv")
icp = icp.dropna(subset=["Country Code"])          # drop footer rows, if any

YEARS = ["2017 [YR2017]", "2021 [YR2021]"]
icp[YEARS] = icp[YEARS].apply(pd.to_numeric, errors="coerce")   # ".." becomes NaN

print("Indicators in file:")
print(icp["Classification Name"].unique())

# Keep only the price level index rows
pli = icp[icp["Classification Name"] == "Price level index (World = 100)"].copy()
pli["category"] = pli["Series Name"].str.split(":").str[1].str.title()

wanted = {"CAN":"Canada","USA":"USA","FRA":"France","ESP":"Spain","GRC":"Greece",
          "GBR":"UK","CHE":"Switzerland","TUR":"Turkey","JPN":"Japan",
          "THA":"Thailand","CHN":"China","ZAF":"South Africa",
          "AUS":"Australia","NZL":"New Zealand"}

present = set(pli["Country Code"].unique())
print("\nMissing countries:", [v for k, v in wanted.items() if k not in present])
print("Extra codes in file:", sorted(present - set(wanted)))

# How many of the 6 categories have a value, per country and year
counts = (pli[pli["Country Code"].isin(wanted)]
          .groupby("Country Name")[YEARS].count())
print("\nCategories with data (max 6):")
print(counts.to_string())

# Quick look: price levels by country, 2021
table = (pli[pli["Country Code"].isin(wanted)]
         .pivot_table(index="Country Name", columns="category",
                      values="2021 [YR2021]"))
print(table.round(1).to_string())

Indicators in file:
['Price level index (World = 100)'
 'Expenditure (local currency units, billions)'
 'Expenditure component share of AIC (AIC = 100%)'
 'Expenditure component share of GDP (GDP = 100%)'
 'Expenditure per capita index, market exchange rate-based (United States= 100)'
 'Expenditure per capita index, market exchange rate-based (World = 100)'
 'Expenditure per capita index, PPP-based (United States = 100)'
 'Expenditure per capita index, PPP-based (World = 100)'
 'Expenditure per capita, market exchange rate-based (US$)'
 'Expenditure per capita, PPP-based (US$)'
 'Expenditure, market exchange rate-based (US$, billions)'
 'Expenditure, PPP-based (US$, billions)' 'Population'
 'Purchasing power parity (PPP) (US$ = 1)'
 'Share of global expenditure, market exchange rate-based (World = 100%)'
 'Share of global expenditure, PPP-based (World = 100%)']

Missing countries: []
Extra codes in file: []

Categories with data (max 6):
                2017 [YR2017]  2021 [YR2021]
Cou

In [3]:
cur = {"CAN":"CAD","USA":"USD","FRA":"EUR","ESP":"EUR","GRC":"EUR","GBR":"GBP",
       "CHE":"CHF","TUR":"TRY","JPN":"JPY","THA":"THB","CHN":"CNY","ZAF":"ZAR",
       "AUS":"AUD","NZL":"NZD"}

tidy = (pli[pli["Country Code"].isin(cur)]
        .melt(id_vars=["Country Name","Country Code","category"],
              value_vars=YEARS, var_name="year", value_name="pli"))

# make sure years are correct type
tidy["year"] = tidy["year"].str[:4].astype(int)
tidy["currency"] = tidy["Country Code"].map(cur)

# Price level relative to Canada (Canada = 100)
can = tidy[tidy["Country Code"]=="CAN"].set_index(["category","year"])["pli"]
tidy["vs_canada"] = [100*p/can[(c, y)] for p, c, y in
                     zip(tidy["pli"], tidy["category"], tidy["year"])]

tidy.to_csv("icp_pli_tidy.csv", index=False)

print(tidy[tidy.year==2017]
      .pivot_table(index="Country Name", columns="category", values="vs_canada")
      .round(0).to_string())

category        Actual Recreation And Culture  Food And Non-Alcoholic Beverages  Households And Npishs Final Consumption Expenditure  Restaurants And Hotels  Transport  Transport Services
Country Name                                                                                                                                                                               
Australia                               107.0                             111.0                                                120.0                   109.0      107.0                91.0
Canada                                  100.0                             100.0                                                100.0                   100.0      100.0               100.0
China                                    59.0                              84.0                                                 59.0                    50.0       62.0                34.0
France                                   98.0               